# Practica 3 NLP

## Importación librerías

In [5]:
!pip uninstall -y transformers
!pip install transformers==4.52.2

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 113.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 

In [1]:
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForQuestionAnswering, pipeline
import torch

## Cargamos Tokenizador y Modelo


In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad', use_fast=True)
model = BertForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you ar

In [3]:
from transformers import pipeline

question_answering = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)

context = """
Artificial intelligence is a field of study that focuses on the creation of systems that can perform tasks that,
normally require human intelligence. These tasks include pattern recognition, decision making, natural language processing, among others,
natural language processing, among others.
"""

question = "What tasks can perform the Artificial Intelligence?"

result = question_answering(
    question=question,
    context=context
)

print(f"Pregunta: {question}")
print(f"Respuesta: {result['answer']}")

Device set to use cuda:0


Pregunta: What tasks can perform the Artificial Intelligence?
Respuesta: pattern recognition, decision making, natural language processing,


## Cargamos el conjunto de datos.


In [4]:
url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json"
!wget $url -O train-v2.0.json

--2026-07-29 09:09:54--  https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json
Resolving rajpurkar.github.io (rajpurkar.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to rajpurkar.github.io (rajpurkar.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42123633 (40M) [application/json]
Saving to: ‘train-v2.0.json’

train-v2.0.json     100%[===================>]  40.17M   157MB/s    in 0.3s    

2026-07-29 09:09:56 (157 MB/s) - ‘train-v2.0.json’ saved [42123633/42123633]



In [5]:
import json

with open('train-v2.0.json') as f:
    squad_data = json.load(f)

## Preparamos los datos.


In [6]:
contexts = []
questions = []
answers = []

for group in squad_data['data']:
    for paragraph in group['paragraphs']:
        context = paragraph['context']
        for qa in paragraph['qas']:
            question = qa['question']
            if 'answers' in qa and len(qa['answers']) > 0:
                answer = qa['answers'][0]#['text']
                contexts.append(context)
                questions.append(question)
                answers.append(answer)

In [7]:
print(f"Number of contexts: {len(contexts)}")
print(f"Number of questions: {len(questions)}")
print(f"Number of answers: {len(answers)}")

Number of contexts: 86821
Number of questions: 86821
Number of answers: 86821


Como el conjunto de datos contiene una gran cantidad de información, seleccionamos únicamente los 100 primeros registros para reentrenar el modelo con el objetivo de que el proceso de entrenamiento sea más rápido.

In [8]:
n_rows = 100
train_contexts = contexts[0:n_rows]
train_questions = questions[0:n_rows]
train_answers = answers[0:n_rows]

n_rows = 25
eval_contexts = contexts[1000:1000+n_rows]
eval_questions = questions[1000:1000+n_rows]
eval_answers = answers[1000:1000+n_rows]

Es necesario tokenizar los textos para después poder entrenar nuestro modelo. Además, para poder evaluar después, necesitamos saber el número de caracter en el que podemos encontrar la respuesta en el contexto.

In [9]:
def find_start_end_tokens(context, answer_text, tokenizer):

    context_tokens = tokenizer.tokenize(context)
    answer_tokens = tokenizer.tokenize(answer_text)

    # Convert tokens to IDs
    context_ids = tokenizer.convert_tokens_to_ids(context_tokens)
    answer_ids = tokenizer.convert_tokens_to_ids(answer_tokens)

    # Find the start and end tokens
    start_idx = -1
    end_idx = -1
    for i in range(len(context_ids) - len(answer_ids) + 1):
        if context_ids[i:i+len(answer_ids)] == answer_ids:
            start_idx = i
            end_idx = i + len(answer_ids) - 1
            break

    return start_idx, end_idx

def add_token_positions(encodings, contexts, answers, tokenizer):
    start_positions = []
    end_positions = []
    for i in range(len(answers)):
        start, end = find_start_end_tokens(contexts[i], answers[i]['text'], tokenizer)

        start_positions.append(start)
        end_positions.append(end)
    encodings.update({'start_positions': start_positions, 'end_positions': end_positions})
    return encodings


In [10]:
encodings = tokenizer(train_contexts, train_questions, truncation=True, padding=True)
encodings = add_token_positions(encodings, train_contexts, train_answers, tokenizer)

eval_encodings = tokenizer(eval_contexts, eval_questions, truncation=True, padding=True)
eval_encodings = add_token_positions(eval_encodings, eval_contexts, eval_answers, tokenizer)

## Definición de Dataset de Torch


In [11]:
# Define the dataset
class SquadDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings.input_ids)

train_dataset = SquadDataset(encodings)
eval_dataset = SquadDataset(eval_encodings)

## Entrenamiento


In [13]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

N_EPOCHS = 1

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.train()

optim = AdamW(model.parameters(), lr=5e-5)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

for epoch in range(N_EPOCHS):
    for batch in train_loader:
        optim.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        start_positions = batch['start_positions'].to(device)
        end_positions = batch['end_positions'].to(device)

        outputs = model(input_ids,
                        attention_mask=attention_mask,
                        start_positions=start_positions,
                        end_positions=end_positions)
        loss = outputs.loss
        loss.backward()
        optim.step()

    print(f"Epoch {epoch+1} completed with loss: {loss.item()}")


question_answering = pipeline('question-answering', model=model, tokenizer=tokenizer)
eval_context = """
Python programming has become very popular due to its simplicity and the wide variety of libraries available for
various applications such as data analysis, artificial intelligence, and web development.
"""
eval_question = "Why is Python programming popular?"

result = question_answering(question=eval_question, context=eval_context)
print(f"Question: {eval_question}")
print(f"Answer: {result['answer']}")

Device set to use cuda:0


Epoch 1 completed with loss: 3.8013248443603516
Question: Why is Python programming popular?
Answer: its simplicity


## Evaluación del modelo



In [15]:
eval_loader = DataLoader(eval_dataset, batch_size=4, shuffle=True)
model.eval()
for epoch in range(1):
    for batch in eval_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        start_positions = batch['start_positions'].to(device)
        end_positions = batch['end_positions'].to(device)

        outputs = model(input_ids,
                        attention_mask=attention_mask,
                        start_positions=start_positions,
                        end_positions=end_positions)
        loss = outputs.loss
        loss.backward()
        optim.step()

    print(f"Epoch {epoch+1} completed with loss: {loss.item()}")


question_answering = pipeline('question-answering', model=model, tokenizer=tokenizer)
eval_context = """
Python programming has become very popular due to its simplicity and the wide variety of libraries available for
various applications such as data analysis, artificial intelligence, and web development.
"""
eval_question = "Why is Python programming popular?"

result = question_answering(question=eval_question, context=eval_context)
print(f"Question: {eval_question}")
print(f"Answer: {result['answer']}")

Device set to use cuda:0


Epoch 1 completed with loss: 1.24161958694458
Question: Why is Python programming popular?
Answer: to its simplicity and the wide variety of libraries


In [16]:
predictions = []
references = []
for idx, context in enumerate(eval_contexts):
    answer = eval_answers[idx]['text']
    question = eval_questions[idx]
    # print(answer)
    # print(question)
    # print(context)
    prediction = question_answering(question=question, context=context)
    predictions.append({"id": idx,"prediction_text": prediction['answer']})
    references.append({"id": idx, "answers": {"text": [answer], "answer_start": [eval_answers[idx]['answer_start']]}})


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [17]:
from collections import Counter

def compute_exact(a_gold, a_pred):
    return int(a_gold == a_pred)

def compute_f1(a_gold, a_pred):
    gold_tokens = a_gold.split()
    pred_tokens = a_pred.split()
    common = Counter(gold_tokens) & Counter(pred_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

score = 0

for pred, ref, question in zip(predictions, references, eval_questions):
    print("---"*10)
    print(f"Question:         {question}")
    print(f"Prediction:       {pred['prediction_text']}")
    print(f"Label:            {ref['answers']['text'][0]}")
    score += compute_f1(pred['prediction_text'], ref['answers']['text'][0])
score / len(eval_contexts)

------------------------------
Question:         What nationality were the two friends who served as a pivotal influence in Frédéric's life while in Paris?
Prediction:       Two
Label:            Polish
------------------------------
Question:         Julian Fontana tried to find his way where before moving to Paris?
Prediction:       in
Label:            England
------------------------------
Question:         Where did Julian Fontana fail to get established?
Prediction:       in
Label:            England
------------------------------
Question:         Who did Chopin know that became rich in Paris?
Prediction:       England; Albert Grzymała,
Label:            Albert Grzymała
------------------------------
Question:         Which friend of Chopin became like an older brother to him?
Prediction:       England; Albert Grzymała,
Label:            Albert Grzymała
------------------------------
Question:         Which friend took on the role of several jobs to help Chopin including copyist

0.35029503829503833

Tras realizar el fine-tuning, el modelo obtuvo una puntuación F1 aproximada de 0,35. Se observan respuestas correctas o parcialmente correctas, aunque en algunos casos el modelo genera fragmentos más extensos que la respuesta esperada. El resultado podría mejorarse aumentando el número de épocas de entrenamiento y utilizando una mayor cantidad de datos.